# 02b — Çok-Seed + Parametre Taraması (Multi-Seed & Parameter Sweep)

**Cold-Start Sosyal-Grafik Sızıntısı — Robustluk Eki**

Bu defter, makaledeki *tek-seed* sonucu, iddianın ("emergent property") gerektirdiği **dağılıma** dönüştürür:

1. **Çok-seed bandı** — taban konfigürasyonda (private 0.30, homophily 0.70, reciprocity 0.49) çok sayıda seed üzerinde cold/warm precision, recall ve saldırgan avantajının ortalaması ± güven aralığı.
2. **OFAT parametre taraması** — `private_ratio`, `homophily`, `reciprocity` parametrelerini teker teker (one-factor-at-a-time) süpürerek etkinin grafik parametrelerinden bağımsızlığını test eder.

**Tasarım sadakati:** Model kodu (`LightGCN`, `FeatureGNN`, `train_bpr`, `link_auc`, gated `Recommender`) 02'den **birebir** kopyalanmıştır. Döngüsellik kuralı korunur: hiçbir yere "hedefin komşusunu öner" kuralı konmaz; sızıntı yine cold-start bağlam-fallback'inden kendiliğinden doğar.

**Kendi kendine yeterli:** 01'in Drive verisine bağımlı değildir; grafikleri 01'in şemasıyla (`SocialGraphDataset`) içeride üretir; böylece parametreleri serbestçe süpürebiliriz. *(Kendi 01 üreticini kullanmak istersen `make_sbm` / `make_lfr` çağrılarını onunla değiştir.)*

> **Çıktı:** Çalışma bitince hücre, Claude'a **doğrudan yapıştırabileceğin** kompakt bir özet (`SHARE_WITH_CLAUDE.md`) basar ve tüm ham satırları `results_long.csv` olarak kaydeder.

## 0. Kurulum

In [ ]:
!pip install -q networkx numpy pandas scipy torch

In [ ]:
import os, json, time, math
import numpy as np, pandas as pd, networkx as nx
import torch, torch.nn as nn
from scipy.stats import wilcoxon

def _in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/cold_start_leakage'
else:
    ROOT = os.path.abspath('./cold_start_leakage')
OUT_DIR = os.path.join(ROOT, 'sweeps')
os.makedirs(OUT_DIR, exist_ok=True)
print("Çıktı dizini:", OUT_DIR)
print("torch", torch.__version__)

## 1. Tarama ayarları

`MODE = "quick"` önce hızlı bir duman testi (birkaç dakika) içindir; her şeyin döndüğünü doğrula, sonra `MODE = "full"` yap. Aşağıdaki gridleri serbestçe değiştirebilirsin.

In [ ]:
MODE = "quick"          # "quick" (hızlı test) | "full" (makaleye giren koşu)

BASELINE = dict(private_ratio=0.30, homophily=0.70, reciprocity=0.49)

# OFAT: her parametre tek tek süpürülür; taban değer band'dan enjekte edilir
OFAT_GRID = {
    "private_ratio": [0.20, 0.40],
    "homophily":     [0.50, 0.90],
    "reciprocity":   [0.30, 0.70],
}

if MODE == "quick":
    MS_SEEDS    = [0, 1, 2, 3]
    OFAT_SEEDS  = [0, 1, 2]
    GRAPHS      = ["sbm"]
    MS_MODELS   = ["lightgcn"]
    EPOCHS      = 30
else:  # full
    MS_SEEDS    = list(range(10))          # 10 seed bandı
    OFAT_SEEDS  = [0, 1, 2, 3, 4]          # 5 seed / OFAT noktası
    GRAPHS      = ["sbm", "lfr"]
    MS_MODELS   = ["lightgcn", "featuregnn"]
    EPOCHS      = 45                        # AUC ~20 epoch'ta doyar; 45 bol bol yeter

K            = 20      # precision/recall@K
K_GATE       = 5       # alpha(h)=h/(h+k_gate) — 02 ile aynı
H_WARM       = 15      # warm gözlemcinin (alakasız) bağlantı sayısı — 02 demo ile aynı
FEAT_DIM     = 16
N_NODES      = 700
print(f"MODE={MODE} | multiseed={len(MS_SEEDS)} seed × {len(GRAPHS)} grafik × {len(MS_MODELS)} model")

## 2. Grafik üretimi (01 şeması; parametrik)

Yönlü takip grafiği: yönsüz taban (SBM/LFR) → `reciprocity` ile yönlendirme (karşılıklı/tek-yön). Özellik vektörü `homophily` ile topluluğa bağlanır; `private_ratio` kadar düğüm gizli etiketlenir. `N(t)=takipçiler ∪ takip edilenler` tam olarak bilinir (ground truth).

In [ ]:
class SocialGraphDataset:
    def __init__(s, name, G, private, community, node_list, features, meta=None):
        s.name, s.G, s.private, s.community = name, G, private, community
        s.node_list, s.features, s.meta = node_list, features, (meta or {})
    def followers(s, t):  return set(s.G.predecessors(t))
    def following(s, t):  return set(s.G.successors(t))
    def neighborhood(s, t): return s.followers(t) | s.following(t)   # GROUND TRUTH

def _orient(Gund, reciprocity, rng):
    D = nx.DiGraph(); D.add_nodes_from(Gund.nodes())
    for u, v in Gund.edges():
        if u == v: continue
        if rng.random() < reciprocity:
            D.add_edge(u, v); D.add_edge(v, u)
        else:
            (D.add_edge(u, v) if rng.random() < 0.5 else D.add_edge(v, u))
    return D

def _features(n, n_comm, comm, homophily, feat_dim, rng):
    mu = rng.normal(size=(n_comm, feat_dim))
    X = np.zeros((n, feat_dim))
    for i in range(n):
        X[i] = homophily * mu[comm[i]] + (1 - homophily) * rng.normal(size=feat_dim)
    X = (X - X.mean(0)) / (X.std(0) + 1e-9)
    return X.astype(np.float32)

def _labels(n, private_ratio, rng):
    k = int(round(private_ratio * n))
    priv = set(rng.choice(n, size=k, replace=False).tolist())
    return {i: (i in priv) for i in range(n)}

def make_sbm(n=700, n_comm=7, p_in=0.06, p_out=0.006,
             reciprocity=0.49, homophily=0.70, private_ratio=0.30,
             feat_dim=16, seed=0):
    rng = np.random.default_rng(seed)
    sizes = [n // n_comm] * n_comm
    for i in range(n - sum(sizes)): sizes[i] += 1
    probs = np.full((n_comm, n_comm), p_out); np.fill_diagonal(probs, p_in)
    Gund = nx.stochastic_block_model(sizes, probs, seed=int(seed))
    comm = {i: Gund.nodes[i]['block'] for i in Gund.nodes()}
    D = _orient(Gund, reciprocity, rng)
    X = _features(n, n_comm, comm, homophily, feat_dim, rng)
    return SocialGraphDataset("sbm", D, _labels(n, private_ratio, rng), comm,
                              list(range(n)), X, {"directed_edges": D.number_of_edges()})

def make_lfr(n=700, reciprocity=0.49, homophily=0.70, private_ratio=0.30,
             feat_dim=16, seed=0, mu=0.1):
    rng = np.random.default_rng(seed); Gund = None
    for off in range(8):
        try:
            Gund = nx.LFR_benchmark_graph(n=n, tau1=3, tau2=1.5, mu=mu,
                    average_degree=10, min_community=30, seed=int(seed)+off, max_iters=500)
            break
        except Exception:
            continue
    if Gund is None:
        return None
    Gund = nx.Graph(Gund); Gund.remove_edges_from(nx.selfloop_edges(Gund))
    Gund = nx.convert_node_labels_to_integers(Gund)
    raw = {i: frozenset(Gund.nodes[i]["community"]) for i in Gund.nodes()}
    uniq = {c: j for j, c in enumerate(set(raw.values()))}
    comm = {i: uniq[raw[i]] for i in Gund.nodes()}; n_comm = len(uniq)
    D = _orient(Gund, reciprocity, rng)
    X = _features(n, n_comm, comm, homophily, feat_dim, rng)
    return SocialGraphDataset("lfr", D, _labels(n, private_ratio, rng), comm,
                              list(range(n)), X, {"directed_edges": D.number_of_edges()})

## 3. Model kodu — **02'den birebir** (LightGCN / FeatureGNN / BPR / link-AUC / Recommender)

In [ ]:
def norm_adj(G, n):
    und = G.to_undirected()
    A = torch.zeros((n, n))
    for u, v in und.edges():
        A[u, v] = 1.0; A[v, u] = 1.0
    A += torch.eye(n)
    deg = A.sum(1); dinv = torch.pow(deg, -0.5); dinv[torch.isinf(dinv)] = 0
    Dm = torch.diag(dinv)
    return Dm @ A @ Dm

class LightGCN(nn.Module):
    def __init__(self, n, dim=64, layers=3):
        super().__init__()
        self.emb = nn.Embedding(n, dim); nn.init.normal_(self.emb.weight, std=0.1)
        self.layers = layers
    def forward(self, Ahat):
        e = self.emb.weight; outs = [e]
        for _ in range(self.layers):
            e = Ahat @ e; outs.append(e)
        return torch.stack(outs, 0).mean(0)

class FeatureGNN(nn.Module):
    def __init__(self, n, feat_dim, dim=64, layers=2):
        super().__init__()
        self.id_emb = nn.Embedding(n, dim); nn.init.normal_(self.id_emb.weight, std=0.1)
        self.feat_proj = nn.Linear(feat_dim, dim)
        self.W = nn.ModuleList([nn.Linear(2*dim, dim) for _ in range(layers)])
        self.act = nn.ReLU(); self.layers = layers
    def forward(self, Ahat, Xfeat):
        h = self.id_emb.weight + self.feat_proj(Xfeat)
        for l in range(self.layers):
            agg = Ahat @ h
            h = self.act(self.W[l](torch.cat([h, agg], 1)))
        return h

def train_bpr(model, Ahat, pos_pairs, n_nodes, Xfeat=None, epochs=60, bs=1024, lr=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    pos = torch.tensor(pos_pairs, dtype=torch.long)
    for ep in range(epochs):
        perm = torch.randperm(pos.size(0))
        for i in range(0, pos.size(0), bs):
            idx = perm[i:i+bs]; u, p = pos[idx, 0], pos[idx, 1]
            neg = torch.randint(0, n_nodes, (u.size(0),))
            E = model(Ahat) if Xfeat is None else model(Ahat, Xfeat)
            su = (E[u]*E[p]).sum(1); sn = (E[u]*E[neg]).sum(1)
            loss = -torch.log(torch.sigmoid(su - sn) + 1e-9).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (model(Ahat) if Xfeat is None else model(Ahat, Xfeat)).detach()

def link_auc(E, G, n_samples=2000, rng=None):
    rng = rng or np.random.default_rng(0)
    und = G.to_undirected(); edges = list(und.edges()); nodes = list(G.nodes())
    pos = [edges[i] for i in rng.choice(len(edges), size=min(n_samples, len(edges)), replace=False)]
    En = torch.nn.functional.normalize(E, dim=1)
    sc = lambda a, b: float((En[a]*En[b]).sum())
    ps = np.array([sc(u, v) for u, v in pos]); ns = []
    while len(ns) < len(ps):
        a, b = rng.choice(nodes, 2, replace=False)
        if not und.has_edge(a, b): ns.append(sc(a, b))
    ns = np.array(ns)
    return float((ps[:, None] > ns[None, :]).mean())

class Recommender:
    def __init__(self, E, node_list, k_gate=5):
        self.E = torch.nn.functional.normalize(torch.as_tensor(E, dtype=torch.float32), dim=1)
        self.node_list = node_list; self.k_gate = k_gate
    def alpha(self, h): return h / (h + self.k_gate)
    def observer_embedding(self, contacts):
        if len(contacts) == 0: return None
        return torch.nn.functional.normalize(self.E[list(contacts)].mean(0, keepdim=True), dim=1)[0]
    def suggest(self, target, observer_contacts, k=20, exclude=None):
        h = len(observer_contacts); a = self.alpha(h)
        ctx = self.E @ self.E[target]
        if h > 0:
            obs = self.E @ self.observer_embedding(observer_contacts)
            score = a*obs + (1-a)*ctx
        else:
            score = ctx
        score = score.clone(); score[target] = -1e9
        for x in (exclude or []): score[x] = -1e9
        return torch.topk(score, k).indices.tolist()

## 4. Ölçüm — cold vs warm + 3 temel (Tablo II'yi yansıtır)

Her gizli hedef için cold/warm precision@K, cold recall@K; rastgele/popülerlik/aynı-topluluk temellerine karşı avantaj; ve cold-vs-warm için eşleştirilmiş Wilcoxon p-değeri.

In [ ]:
def measure(ds, E, k=K, k_gate=K_GATE, h_warm=H_WARM, seed=0):
    rec = Recommender(E, ds.node_list, k_gate=k_gate)
    rng = np.random.default_rng(seed + 9991)
    nodes = ds.node_list
    targets = [t for t in nodes if ds.private[t] and ds.G.degree(t) >= 4]
    deg = dict(ds.G.degree()); deg_sorted = sorted(nodes, key=lambda x: deg[x], reverse=True)
    by_comm = {}
    for x in nodes: by_comm.setdefault(ds.community[x], []).append(x)
    cold_p, cold_r, warm_p, b_rand, b_pop, b_comm = [], [], [], [], [], []
    for t in targets:
        Nt = ds.neighborhood(t)
        if not Nt: continue
        s_cold = set(rec.suggest(t, set(), k=k))
        cold_p.append(len(s_cold & Nt)/k); cold_r.append(len(s_cold & Nt)/len(Nt))
        wc = set(rng.choice(nodes, h_warm, replace=False).tolist()) - Nt - {t}
        warm_p.append(len(set(rec.suggest(t, wc, k=k)) & Nt)/k)
        rr = set(rng.choice(nodes, k, replace=False).tolist()) - {t}
        b_rand.append(len(rr & Nt)/k)
        b_pop.append(len(set([x for x in deg_sorted if x != t][:k]) & Nt)/k)
        same = [x for x in by_comm[ds.community[t]] if x != t]
        ss = set(rng.choice(same, min(k, len(same)), replace=False).tolist()) if same else set()
        b_comm.append(len(ss & Nt)/k)
    cp, wp = np.array(cold_p), np.array(warm_p)
    try:
        p = wilcoxon(cp, wp).pvalue if len(cp) and np.any(cp != wp) else float('nan')
    except Exception:
        p = float('nan')
    best_base = max(np.mean(b_rand), np.mean(b_pop), np.mean(b_comm))
    return dict(n_targets=len(cold_p), cold_prec=float(np.mean(cold_p)),
                cold_recall=float(np.mean(cold_r)), warm_prec=float(np.mean(wp)),
                base_random=float(np.mean(b_rand)), base_pop=float(np.mean(b_pop)),
                base_comm=float(np.mean(b_comm)),
                advantage=float(np.mean(cp) - best_base), wilcoxon_p=float(p))

def run_one(graph_type, params, model_name, seed, epochs=EPOCHS, feat_dim=FEAT_DIM):
    torch.manual_seed(seed); np.random.seed(seed)
    ds = (make_sbm(n=N_NODES, seed=seed, feat_dim=feat_dim, **params) if graph_type == "sbm"
          else make_lfr(n=N_NODES, seed=seed, feat_dim=feat_dim, **params))
    if ds is None: return None
    n = len(ds.node_list)
    Ahat = norm_adj(ds.G, n)
    pos_pairs = [(u, v) for u, v in ds.G.to_undirected().edges()]
    if model_name == "lightgcn":
        mdl = LightGCN(n, dim=64, layers=3); E = train_bpr(mdl, Ahat, pos_pairs, n, epochs=epochs)
    else:
        Xf = torch.tensor(ds.features, dtype=torch.float32)
        mdl = FeatureGNN(n, ds.features.shape[1], dim=64, layers=2)
        E = train_bpr(mdl, Ahat, pos_pairs, n, Xfeat=Xf, epochs=epochs)
    auc = link_auc(E, ds.G, rng=np.random.default_rng(seed))
    m = measure(ds, E.numpy(), seed=seed)
    row = dict(graph_type=graph_type, model=model_name, seed=seed, link_auc=round(auc, 4),
               directed_edges=ds.meta.get("directed_edges"), **params)
    row.update(m); return row

## 5. Süre tahmini (ETA) ve taramayı çalıştır

Önce tek koşuyu zamanlar, toplam süreyi tahmin eder ve sonra tüm taramayı yürütür. İlerleme satır satır basılır; koşu çökerse `results_long.csv` o ana dek biriken satırları korur.

In [ ]:
# tek koşu kalibrasyonu -> ETA
_t = time.time(); _ = run_one(GRAPHS[0], BASELINE, MS_MODELS[0], MS_SEEDS[0]); per_run = time.time() - _t

ms_runs   = len(MS_SEEDS) * len(GRAPHS) * len(MS_MODELS)
ofat_runs = sum(len(v) for v in OFAT_GRID.values()) * len(OFAT_SEEDS) * len(GRAPHS)
total = ms_runs + ofat_runs
print(f"~{per_run:.1f}s/koşu | multiseed={ms_runs} + OFAT={ofat_runs} = {total} koşu")
print(f"Tahmini toplam süre: ~{per_run*total/60:.1f} dakika")

In [ ]:
rows, done, t0 = [], 0, time.time()

def _log(r, total):
    global done; done += 1
    print(f"[{done:3d}/{total}] {r['graph_type']:4s} {r['model']:10s} seed={r['seed']} "
          f"| {[f'{k}={r[k]}' for k in ('private_ratio','homophily','reciprocity')]} "
          f"auc={r['link_auc']:.3f} cold={r['cold_prec']:.3f} warm={r['warm_prec']:.3f} "
          f"recall={r['cold_recall']:.3f} adv={r['advantage']:+.3f} p={r['wilcoxon_p']:.1e}")

total = ms_runs + ofat_runs

# (A) çok-seed bandı — taban konfig
for seed in MS_SEEDS:
    for gt in GRAPHS:
        for mdl in MS_MODELS:
            r = run_one(gt, dict(BASELINE), mdl, seed)
            if r: r["block"] = "multiseed"; r["swept_param"] = "none"; rows.append(r); _log(r, total)
            pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "results_long.csv"), index=False)

# (B) OFAT — her parametre tek tek (LightGCN; baskın yapısal kanal)
for param, values in OFAT_GRID.items():
    for v in values:
        cfg = dict(BASELINE); cfg[param] = v
        for seed in OFAT_SEEDS:
            for gt in GRAPHS:
                r = run_one(gt, cfg, "lightgcn", seed)
                if r: r["block"] = "ofat"; r["swept_param"] = param; rows.append(r); _log(r, total)
                pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "results_long.csv"), index=False)

df = pd.DataFrame(rows)
print(f"\nBitti: {len(df)} satır, {time.time()-t0:.1f}s -> {OUT_DIR}/results_long.csv")
df.head()

## 6. Toplulaştır ve **Claude'a yapıştırılacak özeti** üret

Aşağıdaki hücre iki tablo üretir:
- **Çok-seed bandı:** grafik×model bazında cold/warm/recall/avantaj/AUC ortalaması ± %95 GA (seed'ler arası).
- **OFAT:** parametre×değer×grafik bazında cold precision ortalaması ± std.

Hepsi `SHARE_WITH_CLAUDE.md` dosyasına yazılır **ve** aşağıya basılır. İki `=====` çizgisi arasındaki metni kopyalayıp bana gönderebilirsin.

In [ ]:
def ci95(x):
    x = np.asarray(x, float); n = len(x)
    return 1.96 * x.std(ddof=1) / math.sqrt(n) if n > 1 else float('nan')

df = pd.read_csv(os.path.join(OUT_DIR, "results_long.csv"))
ms = df[df.block == "multiseed"]; of = df[df.block == "ofat"]

# ---- çok-seed bandı ----
band_lines = ["### Çok-seed bandı (taban: private=0.30, homophily=0.70, reciprocity=0.49)",
              "", "| grafik | model | seed | cold_prec (μ±CI) | warm_prec (μ±CI) | cold_recall (μ±CI) | advantage (μ±CI) | link_auc (μ) | medyan p |",
              "|---|---|---|---|---|---|---|---|---|"]
for (gt, mdl), g in ms.groupby(["graph_type", "model"]):
    def mc(col): return f"{g[col].mean():.3f} ± {ci95(g[col]):.3f}"
    band_lines.append(f"| {gt} | {mdl} | {len(g)} | {mc('cold_prec')} | {mc('warm_prec')} | "
                      f"{mc('cold_recall')} | {mc('advantage')} | {g['link_auc'].mean():.3f} | "
                      f"{g['wilcoxon_p'].median():.1e} |")

# ---- OFAT ----
base_pt = {gt: ms[(ms.graph_type == gt) & (ms.model == "lightgcn")] for gt in df.graph_type.unique()}
ofat_lines = ["", "### OFAT parametre taraması (LightGCN; cold_prec μ±std, seed'ler arası)",
              "", "| parametre | değer | grafik | n_seed | cold_prec (μ±std) | warm_prec (μ) | link_auc (μ) |",
              "|---|---|---|---|---|---|---|"]
for param in OFAT_GRID:
    for gt in sorted(of.graph_type.unique()):
        # taban değer noktası (band'dan)
        b = base_pt.get(gt)
        if b is not None and len(b):
            ofat_lines.append(f"| {param} | {BASELINE[param]} (taban) | {gt} | {len(b)} | "
                              f"{b['cold_prec'].mean():.3f} ± {b['cold_prec'].std(ddof=1):.3f} | "
                              f"{b['warm_prec'].mean():.3f} | {b['link_auc'].mean():.3f} |")
        sub = of[(of.swept_param == param) & (of.graph_type == gt)]
        for v, g in sub.groupby(param):
            ofat_lines.append(f"| {param} | {v} | {gt} | {len(g)} | "
                              f"{g['cold_prec'].mean():.3f} ± {g['cold_prec'].std(ddof=1):.3f} | "
                              f"{g['warm_prec'].mean():.3f} | {g['link_auc'].mean():.3f} |")

meta = [f"# Cold-Start — Multi-Seed & Parameter Sweep özeti",
        f"- MODE={MODE} | K={K} | k_gate={K_GATE} | h_warm={H_WARM} | epochs={EPOCHS} | n_nodes={N_NODES}",
        f"- toplam koşu: {len(df)} | grafikler: {sorted(df.graph_type.unique())}", ""]
share = "\n".join(meta + band_lines + ofat_lines) + "\n"

with open(os.path.join(OUT_DIR, "SHARE_WITH_CLAUDE.md"), "w") as f:
    f.write(share)
df.to_csv(os.path.join(OUT_DIR, "results_long.csv"), index=False)

print("=" * 70)
print(share)
print("=" * 70)
print(f"\n[kaydedildi] {OUT_DIR}/SHARE_WITH_CLAUDE.md  ve  results_long.csv")
print("Yukarıdaki iki ===== çizgisi arasını kopyalayıp Claude'a gönder.")

## 7. (Opsiyonel) Görseller — çok-seed kemancı + OFAT eğrileri

In [ ]:
import matplotlib.pyplot as plt
df = pd.read_csv(os.path.join(OUT_DIR, "results_long.csv"))
ms = df[df.block == "multiseed"]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
# kemancı: cold vs warm cold_prec dağılımı (lightgcn)
lg = ms[ms.model == "lightgcn"]
data = [lg["cold_prec"].values, lg["warm_prec"].values]
ax[0].violinplot(data, showmeans=True); ax[0].set_xticks([1, 2]); ax[0].set_xticklabels(["cold", "warm"])
ax[0].set_ylabel("precision@20"); ax[0].set_title(f"Çok-seed dağılımı (LightGCN, n={len(lg)} seed)")
# OFAT homophily eğrisi
of = df[(df.block == "ofat") & (df.swept_param == "homophily")]
for gt, g in of.groupby("graph_type"):
    gg = g.groupby("homophily")["cold_prec"].agg(["mean", "std"]).reset_index()
    ax[1].errorbar(gg["homophily"], gg["mean"], yerr=gg["std"], marker="o", capsize=4, label=gt)
ax[1].set_xlabel("homophily"); ax[1].set_ylabel("cold precision@20"); ax[1].set_title("OFAT: homophily"); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "sweep_figs.png"), dpi=130); plt.show()
print("[kaydedildi]", os.path.join(OUT_DIR, "sweep_figs.png"))